# Fine-tune and Accelerate YOLO26-OBB on AMD Radeon GPU

This hands-on notebook uses **Ultralytics YOLO26** on one **AMD Radeon PRO W7900D** to develop an oriented sheep detector from two prepared aerial datasets. It covers dataset organization, transfer learning, evaluation, accelerated inference, and video tracking.

**Prepared aerial data → COCO-initialized OBB model → fine-tuned detector → ONNX/MIGraphX inference → persistent video tracks**

By the end, you will be able to adapt the same Ultralytics workflow to your own data and models on AMD Radeon GPUs.

## Hands-on roadmap

1. Confirm that Ultralytics can access the AMD Radeon PRO W7900D.
2. Browse both datasets, then inspect the prepared image/label splits and YAML.
3. Load compatible COCO HBB weights into the YOLO26n-OBB architecture.
4. Fine-tune for 8 epochs, with a prepared checkpoint available as fallback.
5. Compare predictions on the validation split and an independently held-out test split.
6. Export to ONNX, verify FP16 MIGraphX inference, and compare accuracy and throughput.
7. Run ByteTrack on held-out aerial video with playback capped at the source FPS.

## 1. Check the environment

Confirm that Ultralytics loads correctly and the AMD GPU is available.

In [ ]:
%env YOLO_OFFLINE=true

import ultralytics

ultralytics.checks()

!rocm-smi --showuse --showmeminfo vram


## 2. Explore the prepared Sheep OBB dataset

The prepared workshop dataset combines two public aerial sources:

| Dataset | Original data | Workshop split |
|---|---|---|
| [SheepCounter v11](https://github.com/idmt-odoll/AUTH-Sheep) | 1,220 train + 507 valid images | 1,220 train; 507 validation |
| [AUTH-Sheep](https://github.com/idmt-odoll/AUTH-Sheep) | 4 annotated UAV videos | `video1` + `video2` + `video4`: 782 training frames; `video3`: 281 test frames |

AUTH-Sheep frames are sampled every 5 frames. Source `sheep` and `goat` annotations are merged into one workshop `sheep` class; all other annotated categories remain background. The result is **2,002 training images**, **507 validation images**, and **281 test frames**. `video3` is absent from training and reused later for inference and full-video tracking.


### Explore the training data

Use the slider below each preview or type an image number directly into its input box. AUTH-Sheep and SheepCounter update independently, and recently viewed samples stay cached.


In [ ]:
from utils.workshop_utils import browse_obb_train_samples

browse_obb_train_samples("/datasets/sheep-datasets")


### Inspect the prepared dataset

The next cell shows the training-ready image and label folders with their file counts, then displays the YAML that connects those splits to the single `sheep` class. Labels use Ultralytics OBB format: `class x1 y1 x2 y2 x3 y3 x4 y4`, with four normalized corner points.


In [ ]:
from pathlib import Path
from utils.workshop_utils import show_dataset_tree

print("Dataset directory tree:\n")
show_dataset_tree("/datasets/sheep-datasets")

print("\n" + "─" * 64 + "\n")
print("Dataset YAML: config/sheep.yaml\n")
print(Path("config/sheep.yaml").read_text())


## 3. Transfer pretrained COCO weights to Sheep OBB

`yolo26n.pt` is an 80-class COCO horizontal-box detector, while `yolo26n-obb.yaml` defines the oriented architecture. Ultralytics loads every shape-compatible weight from the checkpoint. During training, the dataset configuration creates the one-class output head while the OBB-specific layers learn rotated geometry.

This deliberately demonstrates a real downstream change: **general HBB detection → aerial single-class OBB detection**.

### Preview the pretrained COCO model

Use the controls below to browse `sheep` predictions from the pretrained COCO HBB model on familiar COCO images and the new AUTH-Sheep aerial domain. Each title reports the number of detected sheep, and predictions are cached after their first run.


In [ ]:
from ultralytics import YOLO
from utils.workshop_utils import browse_pretrained_coco_predictions

browse_pretrained_coco_predictions(YOLO("/models/yolo26n.pt"))


### Initialize the OBB model

The next cell builds the package-provided YOLO26n-OBB architecture and loads the compatible COCO weights. Training then creates its one-class output head from the dataset configuration.


In [ ]:
import torch

torch.manual_seed(0)
model = YOLO("yolo26n-obb.yaml").load("/models/yolo26n.pt")


## 4. Fine-tune on one W7900D

The three workshop settings stay visible in `model.train()`: training duration, input size, and batch size. `train_defaults.yaml` adds only deterministic training.

On one W7900D, the reference 8-epoch run completes in **about 300 seconds**. Set `run_training = False` only when live training cannot start; the notebook will then load the prepared fine-tuned checkpoint.


In [ ]:
import time
from ultralytics.utils import YAML

run_training = True

if run_training:
    started = time.perf_counter()
    results = model.train(
        data="config/sheep.yaml",
        epochs=8,
        imgsz=768,
        batch=64,
        **YAML.load("config/train_defaults.yaml"),
    )
    best_weights = Path(results.save_dir) / "weights/best.pt"
    print(f"Training wall time: {time.perf_counter() - started:.1f} seconds")
else:
    best_weights = Path("/models/yolo26n_obb_sheep_best.pt")

print("\n" + "─" * 72)
tuned = YOLO(best_weights)
weight_dtype = str(next(tuned.model.parameters()).dtype).replace("torch.bfloat", "BF").replace("torch.float", "FP")
print("Fine-tuned model loaded successfully")
print(f"  Weights       {best_weights}")
print(f"  Task          {tuned.task.upper()}")
print(f"  Weight dtype  {weight_dtype}")
print(f"  Parameters    {sum(parameter.numel() for parameter in tuned.model.parameters()):,}")

## 5. Evaluate the fine-tuned Sheep OBB model

### Quantitative evaluation

The `val` split contains 507 SheepCounter images. The `test` split contains 281 frames sampled every 5 frames from AUTH-Sheep `video3`, which contributes no frame to training. We evaluate the two sources separately with the standard PyTorch FP32 validation path.


In [ ]:
import pandas as pd

sheepcounter = tuned.val(data="config/sheep.yaml", split="val", imgsz=768, batch=1, end2end=False).box
auth_sheep = tuned.val(data="config/sheep.yaml", split="test", imgsz=768, batch=1, end2end=False).box

pd.DataFrame([
    ["val", "SheepCounter", 507, sheepcounter.mp, sheepcounter.mr, sheepcounter.map50, sheepcounter.map],
    ["test", "AUTH-Sheep video3", 281, auth_sheep.mp, auth_sheep.mr, auth_sheep.map50, auth_sheep.map],
], columns=["Split", "Dataset", "Images", "Detection precision", "Recall", "mAP50", "mAP50-95"]).round(3)


### Qualitative before and after fine-tuning

Use the linked controls to view the same held-out aerial frame through both models. The COCO checkpoint predicts its original `sheep` class with horizontal boxes, while the fine-tuned model predicts the workshop's merged sheep/goat class with oriented boxes. This is a qualitative comparison because the models perform different output tasks.


In [ ]:
from utils.workshop_utils import browse_finetuning_comparison

browse_finetuning_comparison(YOLO("/models/yolo26n.pt"), YOLO(best_weights))


## 6. Accelerate Sheep OBB inference with ONNX and MIGraphX

This section exports the fine-tuned model, verifies FP16 MIGraphX inference, compares its accuracy with PyTorch FP16, and measures steady-state throughput.

### Export the fine-tuned OBB model to ONNX

Export a static 768×768 raw OBB graph with `end2end=False`. Ultralytics then applies the same OBB decoding and rotated NMS after PyTorch or MIGraphX inference.

In [ ]:
ONNX_MODEL = Path(YOLO(best_weights).export(
    format="onnx",
    imgsz=768,
    simplify=False,
    end2end=False,
))
print("Exported model:", ONNX_MODEL)

### Run accelerated ONNX inference with MIGraphX

Load the exported ONNX model through `MIGraphXExecutionProvider` in FP16 mode. Ultralytics continues to handle preprocessing, OBB decoding, and rotated NMS. The cell prints a compact inference summary and displays the oriented boxes.

In [ ]:
import onnxruntime as ort

SAMPLE_IMAGE = Path("/datasets/sheep-datasets/images/test") / "auth_v3_00330.jpg"
migraphx = YOLO(ONNX_MODEL, task="obb")
migraphx_result = migraphx.predict(
    SAMPLE_IMAGE, imgsz=768, rect=False, quantize="fp16", end2end=False
)[0]
backend = migraphx.predictor.model.backend

print("\nMIGraphX inference verified")
print("─" * 64)
print(f"Execution path  Ultralytics → ONNX Runtime {ort.__version__} → {backend.provider} → Radeon GPU")
print(f"Model           {ONNX_MODEL.name}")
print(f"Precision       {'FP16' if backend.migraphx_fp16 else 'FP32'}")
print(f"Network input   {' × '.join(map(str, backend.session.get_inputs()[0].shape))}")
print(f"Prediction      {len(migraphx_result.obb)} sheep OBBs · {SAMPLE_IMAGE.name}")
display(migraphx_result.plot(pil=True, labels=False, conf=False, line_width=2))

### Compare PyTorch and MIGraphX accuracy

Compare the optimized FP16 paths: PyTorch uses its default rectangular validation batches, while MIGraphX uses the static 768×768 ONNX input. Both use the same OBB post-processing.


In [ ]:
accuracy_rows = []

for split, dataset, images in (("val", "SheepCounter", 507), ("test", "AUTH-Sheep video3", 281)):
    pytorch_metrics = tuned.val(
        data="config/sheep.yaml", split=split, imgsz=768, batch=1,
        quantize="fp16", end2end=False,
    ).box
    migraphx_metrics = migraphx.val(
        data="config/sheep.yaml", split=split, imgsz=768, batch=1,
        rect=False, quantize="fp16", end2end=False,
    ).box
    accuracy_rows.extend([
        [split, dataset, images, "PyTorch FP16", pytorch_metrics.mp, pytorch_metrics.mr, pytorch_metrics.map50, pytorch_metrics.map],
        [split, dataset, images, "MIGraphX FP16", migraphx_metrics.mp, migraphx_metrics.mr, migraphx_metrics.map50, migraphx_metrics.map],
    ])

pd.DataFrame(accuracy_rows, columns=["Split", "Dataset", "Images", "Backend", "Detection precision", "Recall", "mAP50", "mAP50-95"]).round(3)


### Compare inference performance

Measure steady-state FP16 `YOLO.predict()` throughput at batch 1, 2, and 4. Each backend uses its optimized input path: PyTorch uses rectangular batches, while MIGraphX uses a static 768×768 ONNX export for each batch. Three warmups precede 20 timed predictions on in-memory images.


In [ ]:
import shutil
from ultralytics.utils import LOGGER
from utils.acceleration_utils import plot_backend_benchmark

def measure_throughput(model, batch):
    images = [migraphx_result.orig_img] * batch
    for _ in range(3):
        model.predict(images, imgsz=768, batch=batch, quantize="fp16", end2end=False, verbose=False)
    torch.cuda.synchronize()
    started = time.perf_counter()
    for _ in range(20):
        model.predict(images, imgsz=768, batch=batch, quantize="fp16", end2end=False, verbose=False)
    torch.cuda.synchronize()
    return batch * 20 / (time.perf_counter() - started)

LOGGER.disabled = True
ort.set_default_logger_severity(3)
pytorch = YOLO(best_weights)
export_model = YOLO(shutil.copy(best_weights, "/tmp/yolo26n_obb_sheep_benchmark.pt"))
rows = []
for batch in (1, 2, 4):
    rows.append(["PyTorch", batch, measure_throughput(pytorch, batch)])
    onnx_model = export_model.export(
        format="onnx", imgsz=768, batch=batch, simplify=False, end2end=False
    )
    rows.append(["MIGraphX", batch, measure_throughput(YOLO(onnx_model, task="obb"), batch)])
LOGGER.disabled = False
ort.set_default_logger_severity(2)

plot_backend_benchmark(pd.DataFrame(rows, columns=["Backend", "Batch", "Throughput (images/s)"]))


## 7. Run real-time OBB tracking

OpenCV reads each frame and Ultralytics `model.track()` applies OBB detection and ByteTrack association. `LiveCanvas` encapsulates OBB rendering, limits browser updates, and caps playback at the source frame rate.

The default `video3` is the held-out source behind the sampled test split; here the complete original video is used. Change `VIDEO_NAME` to explore the three training videos.

In [ ]:
import cv2
from utils.live_tracking_demo import LiveCanvas

VIDEO_NAME = "video3"  # Test video; try "video1", "video2", or "video4".
video = cv2.VideoCapture(f"/datasets/sheep-datasets/raw/AUTH-Sheep/{VIDEO_NAME}.mp4")
source_fps = video.get(cv2.CAP_PROP_FPS) or 30.0
track_model = YOLO(best_weights)
canvas = LiveCanvas(fps=source_fps, size=(960, 540))

while video.isOpened():
    success, frame = video.read()
    if not success:
        break

    frame = cv2.resize(frame, canvas.size)
    tracked = track_model.track(
        frame,
        persist=True,
        tracker="config/bytetrack_sheep_workshop.yaml",
        imgsz=768,
        quantize="fp16",
        end2end=False,
        verbose=False,
    )[0]
    canvas.write(tracked)

video.release()
canvas.finish()


## Wrap-up and further exploration

You have completed one practical Ultralytics workflow on an AMD Radeon GPU:

- inspected the dataset layout and YAML;
- initialized OBB from compatible COCO HBB weights;
- fine-tuned or loaded the prepared model and evaluated the validation and test splits;
- exported to ONNX and compared PyTorch with MIGraphX;
- streamed persistent OBB tracks with source-rate pacing.

Try another source video, input size, or tracker configuration. The same compact API connects data, training, evaluation, accelerated inference, and multi-object tracking.